In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium Plan, Flex Plan e On-Prem (via IBM Quantum Platform API) Plan. Elas estão em status de versão prévia e sujeitas a alterações.

## Visão geral
Na química quântica, o problema da estrutura eletrônica consiste em encontrar as soluções para a equação de Schrödinger eletrônica — as funções de onda quânticas que descrevem o comportamento dos elétrons do sistema. Essas funções de onda são vetores de amplitudes complexas, onde cada amplitude corresponde à contribuição de uma possível configuração eletrônica.

O estado fundamental é a função de onda de menor energia do sistema e tem uma importância especial no estudo de sistemas moleculares. A abordagem mais precisa para calcular o estado fundamental considera todas as configurações eletrônicas possíveis, mas isso se torna inviável para sistemas maiores, pois o número de configurações cresce exponencialmente com o tamanho do sistema.

O Handover Iterative Variational Quantum Eigensolver (HI-VQE) é um método híbrido quântico-clássico inovador para estimar com precisão o estado fundamental de sistemas moleculares. Ele integra hardware quântico com computação clássica, usando processadores quânticos para explorar eficientemente configurações eletrônicas candidatas e calculando a função de onda resultante em computadores clássicos. Ao gerar funções de onda compactas e quimicamente precisas, o HI-VQE aprimora a pesquisa e a descoberta em química quântica e ciência dos materiais.

![Imagem mostrando uma visão geral do algoritmo HI-VQE da Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

O HI-VQE reduz a complexidade computacional do problema da estrutura eletrônica ao estimar o estado fundamental com alta precisão de forma eficiente. Ele se concentra em um subconjunto cuidadosamente selecionado das configurações eletrônicas mais relevantes, otimizando tanto a precisão quanto a eficiência.

Combinando os pontos fortes de computadores clássicos e quânticos, o HI-VQE refina e aprimora iterativamente a estimativa atual da função de onda. Suas técnicas únicas de construção de subespaço ajudam a tornar a seleção de configurações mais eficiente, para que os usuários tenham maior controle computacional e maior precisão nas simulações de química quântica.

Se você quiser conhecer o algoritmo com mais profundidade, pode [ler o artigo de pesquisa associado](https://arxiv.org/abs/2503.06292).

## Descrição
O número de configurações eletrônicas para um sistema molecular cresce exponencialmente com o tamanho do sistema. No entanto, para certos estados eletrônicos, como o estado fundamental, é comum que apenas uma pequena fração das configurações contribua de forma significativa para a energia do estado. Os métodos de interação de configuração selecionada (SCI) exploram essa esparsidade para reduzir os custos computacionais, identificando e focando nas configurações mais relevantes. Esse subconjunto de configurações é chamado de subespaço.

O HI-VQE aproveita a eficiência inerente dos computadores quânticos para representar sistemas moleculares e auxiliar na busca de subespaços. Ele integra sub-rotinas clássicas e quânticas para resolver o problema da estrutura eletrônica com alta precisão. Ao contrário dos métodos quânticos SCI existentes, o HI-VQE combina treinamento variacional, construção iterativa de subespaço e triagem de configurações por pré-diagonalização para aumentar a eficiência, reduzindo medições quânticas, iterações e custos de diagonalização clássica. O HI-VQE pode, portanto, ser aplicado a sistemas moleculares maiores que requerem mais qubits, e reduz o custo para resolver um problema de determinado tamanho com o mesmo grau de precisão.

![Imagem mostrando uma descrição detalhada de cada etapa do algoritmo HI-VQE da Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Para calcular o estado fundamental de um sistema, o HI-VQE usa primeiro o pacote de química clássica PySCF para gerar uma representação molecular a partir das entradas fornecidas pelo usuário, como a geometria molecular e outras informações moleculares. Em seguida, entra em um loop de otimização híbrido quântico-clássico, refinando iterativamente um subespaço para representar de forma otimizada o estado fundamental enquanto minimiza o número de configurações incluídas. O loop continua até que critérios de convergência, como tamanho do subespaço ou estabilidade de energia, sejam satisfeitos, após o que a função de onda do estado fundamental calculada e sua energia são produzidas como saída. Esses resultados podem ser usados para construir superfícies de energia potencial precisas e realizar análises mais aprofundadas do sistema.

O loop de otimização se concentra em ajustar os parâmetros de um Circuit quântico para gerar um subespaço de alta qualidade. O HI-VQE oferece três opções de Circuit quântico: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) e [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). A otimização é inicializada próximo ao estado de referência de Hartree-Fock devido à sua adequação geral. O Circuit é então executado em um dispositivo quântico e as configurações são amostradas do estado quântico resultante antes de serem retornadas como strings binárias. Devido ao ruído do dispositivo quântico, algumas configurações amostradas podem ser fisicamente inválidas, não conservando o número de elétrons ou o spin. O HI-VQE resolve isso usando o processo de recuperação de configuração do pacote [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), para que os usuários possam corrigir configurações inválidas ou descartá-las.

As configurações válidas passam então por uma etapa de triagem opcional para remover aquelas que se prevê contribuir minimamente. Isso reduz a dimensão do subespaço, diminuindo assim o custo da etapa de diagonalização. Se a triagem estiver habilitada, um Hamiltoniano de subespaço preliminar é construído a partir das configurações válidas e uma diagonalização é realizada com critérios de encerramento muito relaxados. Embora a precisão das amplitudes resultantes para cada configuração seja baixa, ela é eficaz para prever quais configurações deixar de fora do subespaço nesta iteração, e é rápida de calcular.

As configurações selecionadas são adicionadas ao subespaço e o Hamiltoniano do sistema é projetado nesse subespaço. O subespaço se atualiza iterativamente, preservando as configurações mais relevantes entre as iterações. Essa abordagem contrasta com métodos alternativos porque o Circuit quântico não precisa aproximar o estado fundamental completo a cada etapa.

Em seguida, o Hamiltoniano do subespaço é diagonalizado classicamente para obter o menor autovalor e seu autovetor correspondente, representando uma aproximação do estado fundamental e sua energia. À medida que a qualidade do subespaço melhora com as iterações, o estado fundamental calculado se aproxima melhor do estado fundamental verdadeiro. Uma etapa adicional de triagem pode ser realizada neste ponto para remover quaisquer configurações do subespaço que não tenham uma contribuição substancial para o estado fundamental calculado. Essa etapa garante que o subespaço levado para a próxima iteração seja o mais compacto possível. Isso é avaliado com base nas amplitudes retornadas pela diagonalização, pois elas representam a contribuição de importância de cada configuração para o estado fundamental calculado.

Uma verificação de convergência então determina se um treinamento adicional melhoraria os resultados. Se sim, uma etapa de expansão clássica opcional é realizada, os parâmetros do Circuit quântico são atualizados para minimizar ainda mais a energia calculada e o processo se repete. A etapa de expansão clássica gera configurações adicionais para o subespaço, complementando as configurações amostradas do dispositivo quântico. Ela primeiro identifica a configuração com a maior amplitude nos resultados da diagonalização, antes de gerar novas configurações com excitações simples e duplas a partir da configuração identificada. O número desejado dessas configurações é então adicionado ao subespaço.

Uma vez determinado que as iterações convergiram, o HI-VQE retorna o estado fundamental calculado (na forma dos estados no subespaço e suas amplitudes na função de onda do estado fundamental), sua energia e uma medida de variância de energia que indica se o estado calculado forma um autoestado do Hamiltoniano do sistema.

Os usuários podem decidir o Circuit quântico utilizado e o número de shots tomados para cada Circuit quântico, bem como controlar o tamanho do subespaço ou habilitar a geração clássica de configurações adicionais para auxiliar as configurações geradas pelo quantum. Dessa forma, os usuários podem adaptar o comportamento do HI-VQE para suas aplicações desejadas.

## Primeiros passos
Primeiro, [solicite acesso à função](https://forms.office.com/r/zN3hvMTqJ1).
Em seguida, autentique-se usando sua [chave de API do IBM Quantum&reg;](http://quantum.cloud.ibm.com/) e, assumindo que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local, selecione a Qiskit Function da seguinte forma:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Entradas
Consulte a tabela a seguir para todos os parâmetros de entrada que a função aceita. As seções subsequentes de [Opções de molécula](#molecule-options) e [Opções do HI-VQE](#hi-vqe-options) entram em mais detalhes sobre esses argumentos.

| Nome                   | Tipo                                                           | Descrição                                                                                                                                                                                                                                                                                                 | Obrigatório | Padrão                                                                  | Exemplo                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Pode ser uma string ou listas estruturadas contendo pares de átomo e coordenada. Se fornecido como string, deve ser uma geometria molecular no Formato de Coordenadas Cartesianas. Se fornecido como lista, deve ser uma lista de listas, cada uma contendo uma string de átomo e uma tupla de coordenadas. | Sim      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` ou `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Nome do backend para fazer a consulta.                                                                                                                                                                                                                                                                      | Sim      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | A dimensão máxima do subespaço para a diagonalização. Menos estados serão usados se o número não for um quadrado perfeito.                                                                                                                                                                                  | Sim      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | O número máximo de estados CI gerados classicamente a serem incluídos em cada iteração.                                                                                                                                                                                                                     | Sim      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Opções relacionadas à molécula usada como entrada para o HI-VQE. Consulte a seção [Opções de molécula](#molecule-options) para mais detalhes.                                                                                                                                                               | Não       | Consulte a seção [Opções de molécula](#molecule-options) para detalhes.                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Opções que controlam o comportamento do algoritmo HI-VQE. Consulte a seção [Opções do HI-VQE](#hi-vqe-options) para mais detalhes.                                                                                                                                                                          | Não       | Consulte a seção [Opções do HI-VQE](#hi-vqe-options) para detalhes.                                 | `{"shots": 10_000, "max_iter": 10 }`                               |

### Opções de molécula
A tabela a seguir detalha todas as chaves e valores que podem ser definidos no dicionário `molecule_options`, bem como seus tipos de dados e valores padrão. Todas as chaves são opcionais.

| Chave                             | Tipo do valor                       | Valor padrão                     | Faixa válida                                                                                                                                                   | Explicação                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Variado                                                                                                                                                        | Um inteiro especificando a carga líquida total do sistema molecular. O valor padrão é 0; no entanto, pode ser qualquer inteiro.                                                                                                                                                                                                                                                                                                                                                                                                          |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Variado                                                                                                                                                        | Uma string especificando o tipo de base; elas são passadas ao pyscf. (ex: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                               |
| `"active_orbitals"`               | `List[int]`                         | Todos os índices de orbital.     | Os índices de orbital espacial válidos para o problema.                                                                                                        | Uma lista de índices de orbitais ativos no intervalo [0, n), onde n é o número de qubits usados no problema. Se especificado, o argumento frozen_orbitals também deve ser especificado.                                                                                                                                                                                                                                                                                                                                                  |
| `"frozen_orbitals"`               | `List[int]`                         | Sem índices.                     | Os índices de orbital espacial válidos para o problema, excluindo orbitais ativos.                                                                             | Uma lista de índices de orbitais congelados no mesmo intervalo que active_orbitals. Se especificado, active_orbitals também deve ser especificado. Note que apenas orbitais ocupados devem ser congelados, pois o número de elétrons ativos é reduzido em 2 para cada orbital ocupado que é congelado.                                                                                                                                                                                                                                   |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Orbitais moleculares de Hartree-Fock. | Variado.                                                                                                                                                   | Os coeficientes para os orbitais espaciais usados no cálculo das integrais de repulsão eletrônica do sistema. Alguns exemplos válidos são orbitais moleculares de Hartree-Fock, orbitais naturais e orbitais AVAS.                                                                                                                                                                                                                                                                                                                      |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` ou `False`                                                                                                                                              | Usado para invocar a simetria de grupo pontual para os cálculos moleculares iniciais, a fim de construir a base orbital adaptada à simetria. Esses orbitais adaptados à simetria são usados como funções de base para os seguintes cálculos SCF. O valor padrão é False; se definido como True, será invocado e grupos pontuais arbitrários serão detectados e usados automaticamente. Se uma simetria específica for atribuída, por exemplo, symmetry = "Dooh", um erro será levantado se a geometria molecular não estiver sujeita a essa simetria requerida. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Consulte a [documentação do pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                            | Pode ser usado para gerar um subgrupo da simetria detectada. Isso não tem efeito quando a simetria é especificada usando o argumento de palavra-chave symmetry.                                                                                                                                                                                                                                                                                                                                                                          |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Consulte a [documentação do pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                            | Especifica a unidade de medida a ser usada para coordenadas atômicas e distâncias. O padrão é usar unidades de angstrom.                                                                                                                                                                                                                                                                                                                                                                                                                |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Consulte a [documentação do pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                            | Especifica o modelo nuclear a ser usado. Por padrão, usa o modelo nuclear pontual, e outros valores habilitam o modelo nuclear gaussiano. Se uma função for fornecida, ela será usada com o modelo nuclear gaussiano para gerar o valor de distribuição de carga nuclear 'zeta'.                                                                                                                                                                                                                                                          |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Consulte a [documentação do pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                            | Especifica o pseudopotencial para os átomos na molécula. Por padrão é None, indicando que nenhum pseudopotencial é aplicado e todos os elétrons são explicitamente incluídos nos cálculos.                                                                                                                                                                                                                                                                                                                                               |
| `"cart"`                          | `bool`                              | `False`                          | Consulte a [documentação do pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                            | Especifica se deve usar GTOs cartesianos como funções de base de momento angular no cálculo. O valor padrão False usará GTOs esféricos.                                                                                                                                                                                                                                                                                                                                                                                                  |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Consulte a [documentação do pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                            | Define o momento magnético de spin colinear de cada átomo. Por padrão, é None e cada átomo é inicializado com spin zero.                                                                                                                                                                                                                                                                                                                                                                                                                |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | ex.: ["H 1s", "O 2p"] para H2O                                                                                                                                | Define os Orbitais Atômicos a serem incluídos no esquema AVAS. Consulte a [documentação do AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                     |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Entre 0.0 e 2.0                                                                                                                                               | Especifica o valor de corte usado para determinar quais Orbitais Atômicos (AOs) são retidos no espaço ativo.                                                                                                                                                                                                                                                                                                                                                                                                                            |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` ou `"ccsd"`                                                                                                                                            | Define a abordagem teórica para preparar orbitais naturais e selecionar orbitais ativos com base no esquema de Números de Ocupação de Orbitais Naturais (NOONs). Consulte a [documentação dos NOONs](https://doi.org/10.1063/5.0042147). Tanto os índices de orbitais ativos quanto os congelados devem ser fornecidos para controlar o número de orbitais (e o número de qubits).                                                                                                                                                        |

### Opções do HI-VQE
A tabela a seguir detalha todas as chaves e valores que podem ser definidos no dicionário `hivqe_options`, bem como seus tipos de dados e valores padrão. Todas as chaves são opcionais.

| Chave                             | Tipo do valor                       | Valor padrão                     | Faixa válida                                                                                                                                                   | Explicação                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Entre 1 e 10 000.                                                                                                                                              | Número de shots a usar no dispositivo quântico por iteração.                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
| `"max_iter"`                      | `int`                               | `25`                             | Entre 1 e 50.                                                                                                                                                  | O número máximo de iterações a executar para otimizar o ansatz. O algoritmo pode usar menos iterações se a convergência for alcançada mais cedo.                                                                                                                                                                                                                                                                                                                                                                                         |
| `"initial_basis_states"`          | `List[str]`                         | O estado de Hartree-Fock.        | Strings binárias com o número de bits correspondendo ao número necessário de qubits para o problema.                                                           | Pode ser usado para reiniciar o algoritmo com estados clássicos de um resultado anterior.                                                                                                                                                                                                                                                                                                                                                                                                                                               |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"` ou `"lucj"`                                                                                                                                   | Especifica o ansatz quântico a otimizar para gerar novos estados. `"epa"` seleciona o ansatz de preservação de excitação. `"hea"` seleciona o ansatz eficiente em hardware. `"lucj"` seleciona o ansatz local unitary cluster Jastrow.                                                                                                                                                                                                                                                                                                   |
| `"convergence_count"`             | `int`                               | `3`                              | Pelo menos 2.                                                                                                                                                  | O número de iterações sem alteração significativa da energia calculada que devem decorrer antes de o algoritmo ser considerado convergido.                                                                                                                                                                                                                                                                                                                                                                                               |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Maior que 0 e no máximo 1.                                                                                                                                     | A magnitude da mudança na energia calculada que é considerada significativa para fins de verificações de convergência.                                                                                                                                                                                                                                                                                                                                                                                                                  |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` ou `False`                                                                                                                                              | Se `True`, as iterações de `convergence_count` devem ocorrer sem uma mudança significativa interrompente para qualificar como convergência. Se `False`, o algoritmo parará após `convergence_count` se mudanças insignificantes tiverem ocorrido em quaisquer iterações durante o processo de otimização.                                                                                                                                                                                                                                  |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` ou `False`.                                                                                                                                             | Se deve ou não usar a recuperação de configuração do pacote `qiskit-addon-sqd`. Se True, estados inválidos amostrados do dispositivo quântico são corrigidos classicamente. Se False, eles são descartados.                                                                                                                                                                                                                                                                                                                              |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Qualquer um de `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"` ou `"sca"`. Se usar o ansatz `"lucj"`, `"lucj_default"` também é uma opção. | Especifica o esquema de entrelaçamento a ser usado dentro do Circuit quântico, seguindo as convenções comuns do Qiskit e do [ffsim para o ansatz LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Maior que 0.                                                                                                                                                   | O número de repetições de cada camada no Circuit quântico.                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Pelo menos 0 e menor que 1.                                                                                                                                    | A tolerância para decidir quais estados devem ser triados para fora do subespaço após a diagonalização. Especifica o limiar de inclusão para os estados do subespaço com base em suas amplitudes calculadas.                                                                                                                                                                                                                                                                                                                             |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Entre `1e-4` e `1e-1`, inclusive.                                                                                                                              | A tolerância para prever quais estados devem ser triados para fora do subespaço antes da diagonalização. Controla a precisão das amplitudes previstas para cada estado, com um valor menor resultando em previsões mais precisas.                                                                                                                                                                                                                                                                                                         |

## Saídas
A função retorna um dicionário com quatro chaves e valores. As chaves e valores estão documentados na tabela a seguir:

| Chave           | Tipo do valor | Explicação                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | A energia aproximada do estado fundamental da molécula.                                                                   |
| `"states"`      | `List[str]`   | Os determinantes selecionados que formam o subespaço usado para calcular a energia. Estão no formato alternado alfa-beta. |
| `"eigenvector"` | `List[float]` | O autovetor correspondente ao estado fundamental do subespaço composto por `"states"`.                                    |
| `"energy_variance"` | `float` | A variância de energia do estado fundamental do subespaço composto por `"states"`, indicando a qualidade da solução. Esse valor é não-negativo e um valor menor significa que o estado fundamental do subespaço aproxima mais de perto um autoestado do Hamiltoniano do sistema. |
| `"energy_history"` | `List[float]` | As energias calculadas a cada iteração durante o processo de otimização híbrida, na mesma ordem em que foram calculadas. Duas energias são calculadas por iteração como parte do processo de otimização SPSA. |

## Exemplo
O primeiro exemplo mostra como calcular a energia do estado fundamental de uma molécula de NH3 usando o algoritmo HI-VQE.

#### Definir a geometria molecular e as opções
A geometria molecular do NH3 é fornecida com coordenadas cartesianas separadas por ";" para cada átomo.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Opções adicionais podem ser definidas e fornecidas para o sistema molecular no seguinte formato de dicionário.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Execute a função com as entradas de geometria e opções.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

É uma boa ideia imprimir o ID do job da Function para que ele possa ser fornecido em solicitações de suporte caso algo dê errado.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Este exemplo usa então 16 qubits com 8 orbitais da base sto3g para uma molécula de NH3.
Verifique o [status](/guides/functions#check-job-status) do workload da sua Qiskit Function ou obtenha os [resultados](/guides/functions#retrieve-results) da seguinte forma:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Após a conclusão do job, os resultados podem ser obtidos com a instância `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Para acessar a energia do estado fundamental, use a chave "energy". A chave "eigenvector" fornece os coeficientes CI com a notação de bitstring correspondente da configuração eletrônica armazenada em "states" dos resultados.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Saída:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936

## Desempenho
Esta seção mostra os cálculos de benchmark demonstrados do HI-VQE com um caso de 24 qubits para Li2S, um caso de 40 qubits para uma molécula N2 e um caso de 44 qubits para um sistema FeP-NO.

#### Curva da superfície de energia potencial de dissociação para uma molécula de Li2S com 24 qubits
A curva PES é mostrada com a referência FCI e estimativa inicial de RHF, juntamente com o erro de energia em relação à referência FCI.

![Imagem mostrando que o HI-VQE produz soluções dentro da precisão química de uma curva PES de referência clássica para o sistema Li2S.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Os cálculos foram realizados com as seguintes geometrias e opções.